# 01. Exploración y Auditoría de la API de Mercado Libre Colombia (MCO)

**Fase 1 del Proyecto de Grado / Emprendimiento**  
**Autor:** Julián Duarte  
**Objetivo:** Verificar qué datos entrega realmente la API de Mercado Libre, cuáles requieren autenticación (`access_token`) y cómo se estructuran las publicaciones de Memorias RAM (`MCO1694`).

In [ ]:
import os
import json
import requests
import pandas as pd
from dotenv import load_dotenv
from pathlib import Path

# Cargar credenciales desde .env
load_dotenv()
access_token = os.getenv("MELI_ACCESS_TOKEN")
print(f"Token cargado: {'SI' if access_token else 'NO (Petición anónima)'}")

## 1. Verificación de Categoría Pública de Memorias RAM (`MCO1694`)
Este endpoint es 100% público y no requiere autenticación.

In [ ]:
url_cat = "https://api.mercadolibre.com/categories/MCO1694"
resp_cat = requests.get(url_cat)
data_cat = resp_cat.json()

print("Categoría:", data_cat.get("name"))
print("Total publicaciones activas:", data_cat.get("total_items_in_this_category"))
path_names = [p["name"] for p in data_cat.get("path_from_root", [])]
print("Ruta:", " > ".join(path_names))

## 2. Búsqueda de Publicaciones en Mercado Libre Colombia
> **Nota:** Debido a las políticas de seguridad de Mercado Libre (`PolicyAgent`), este endpoint requiere cabecera de autenticación (`Authorization: Bearer <TOKEN>`).

In [ ]:
headers = {}
if access_token:
    headers["Authorization"] = f"Bearer {access_token}"

search_url = "https://api.mercadolibre.com/sites/MCO/search"
params = {"category": "MCO1694", "limit": 10}

r_search = requests.get(search_url, headers=headers, params=params)
print(f"Status Code: {r_search.status_code}")
if r_search.status_code == 200:
    items = r_search.json().get("results", [])
    df = pd.DataFrame([{
        "id": it.get("id"),
        "title": it.get("title"),
        "price": it.get("price"),
        "sold_quantity": it.get("sold_quantity"),
        "logistic_type": it.get("shipping", {}).get("logistic_type"),
        "free_shipping": it.get("shipping", {}).get("free_shipping")
    } for it in items])
    print(f"Se obtuvieron {len(df)} publicaciones:")
    print(df.head())
else:
    print("Respuesta recibida:", r_search.json())